# breast cancer one-class svm, with probabilities

four ways to turn the svm score into a number from 0 to 1. i want to see which one make sense when someone ask me what 0.83 mean.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar, brentq
from scipy.special import expit
from scipy.stats import gamma
from sklearn.datasets import load_breast_cancer
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)

In [ ]:
bc = load_breast_cancer()
Xall, yall = bc.data, bc.target
# yall: 1 means no cancer here. fine.

normal_idx = np.where(yall == 1)[0]
anomaly_idx = np.where(yall == 0)[0]
rng.shuffle(normal_idx)

cut = int(len(normal_idx) * 0.6)
fit_idx = normal_idx[:cut]
held_normal = normal_idx[cut:]

sc = StandardScaler().fit(Xall[fit_idx])
Xfit = sc.transform(Xall[fit_idx])
Xeval = sc.transform(np.vstack([Xall[held_normal], Xall[anomaly_idx]]))
yeval = np.r_[np.ones(len(held_normal)), np.zeros(len(anomaly_idx))]

len(fit_idx), len(Xeval), yeval.mean().round(3)

In [ ]:
svm = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')
svm.fit(Xfit)
g_fit  = svm.decision_function(Xfit)
g_eval = svm.decision_function(Xeval)
fmax = g_fit.max()

plt.figure(figsize=(6.5, 2.6))
for cls, c in [(1, '#2a6'), (0, '#c33')]:
    plt.hist(g_eval[yeval==cls], bins=24, color=c, alpha=0.55, edgecolor='none')
plt.axvline(0, color='k', lw=0.5)
plt.xlabel('g(x)')
plt.yticks([])
plt.show()

now the four method. same input (svm score on test set), four outputs.

In [ ]:
# Platt. only OCSVM's own sign as pseudo-truth, since there's nothing else at fit time.
# Fit A,B via Nelder-Mead on a 1-D thing wrapped in a 1-D thing -- I just use minimize_scalar on A
# with B closed-form... no, easier: do a tiny 2D Nelder-Mead by hand-ish via scipy.
from scipy.optimize import minimize

def platt(g_train, g_query, pseudo):
    pseudo = pseudo.astype(float)
    Np = pseudo.sum(); Nn = len(pseudo) - Np
    t = np.where(pseudo > 0, (Np+1)/(Np+2), 1/(Nn+2))
    def J(ab):
        z = ab[0]*g_train + ab[1]
        return (t*z + np.log1p(np.exp(-z))).sum() + ((1-t)*0).sum()
    out = minimize(J, [0.0, 0.0], method='Nelder-Mead', options={'xatol':1e-5})
    A, B = out.x
    return expit(-(A*g_query + B))

pseudo = (svm.predict(Xfit) > 0).astype(int)

In [ ]:
EPS = 0.001
K = 5  # bins per side

def _nearest_mark(marks, probs, q):
    j = np.abs(q[:, None] - marks[None, :]).argmin(axis=1)
    return probs[j]

def equidistant(g_train, g_query):
    a = g_train.min(); b = g_train.max()
    left  = np.linspace(a, 0, K+1)[:-1]
    right = np.linspace(0, b, K+1)[1:]
    marks = np.r_[left, 0.0, right]
    ps = np.linspace(EPS, 1-EPS, marks.size)
    return _nearest_mark(marks, ps, g_query)

def density_bin(g_train, g_query):
    qs = (np.arange(K) + 0.5) / K
    neg = g_train[g_train < 0]
    pos = g_train[g_train >= 0]
    nm = np.quantile(neg, qs) if len(neg) else np.zeros(K)
    pm = np.quantile(pos, qs) if len(pos) else np.zeros(K)
    marks = np.r_[nm, 0.0, pm]
    ps = np.linspace(EPS, 1-EPS, marks.size)
    return _nearest_mark(marks, ps, g_query)

In [ ]:
def gamma_scale(g_train, g_query):
    # S = (fmax - g)_+. moment-fit a Gamma on the training Ss, then bend so g=0 -> 0.5.
    S = np.clip(fmax - g_train, 0, None)
    S = S[S > 0]
    m, v = S.mean(), S.var()
    shape, scale = m*m/v, v/m
    Sq = np.clip(fmax - g_query, 0, None)
    surv = 1 - gamma.cdf(Sq, shape, scale=scale)
    surv0 = 1 - gamma.cdf(fmax, shape, scale=scale)
    # huh -- when fmax is tiny this collapses. add a floor.
    surv0 = max(surv0, 1e-9)
    up = 0.5 + 0.5 * (surv - surv0) / max(1 - surv0, 1e-9)
    dn = 0.5 * surv / surv0
    return np.where(surv >= surv0, up, dn).clip(EPS, 1-EPS)

In [ ]:
probs = {
    'platt':       platt(g_fit, g_eval, pseudo),
    'equi-bin':    equidistant(g_fit, g_eval),
    'dens-bin':    density_bin(g_fit, g_eval),
    'gamma-scale': gamma_scale(g_fit, g_eval),
}

# pick three eval rows. random, but reproducible.
trio = rng.choice(len(g_eval), size=3, replace=False)
print('row    g       y     ' + '   '.join(f'{k:>11}' for k in probs))
for r in trio:
    line = f'{r:>3}  {g_eval[r]:+.3f}  {int(yeval[r])}    '
    line += '   '.join(f'{probs[k][r]:>11.3f}' for k in probs)
    print(line)

In [ ]:
# reliability, 10 quantile bins
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0,1],[0,1], color='#999', lw=0.7)
for name, p in probs.items():
    e = np.quantile(p, np.linspace(0, 1, 11))
    e[0] -= 1e-6; e[-1] += 1e-6
    xs, ys = [], []
    for i in range(10):
        m = (p >= e[i]) & (p < e[i+1])
        if m.sum() >= 2:
            xs.append(p[m].mean()); ys.append(yeval[m].mean())
    ax.plot(xs, ys, '.-', label=name, lw=1)
ax.set_xlabel('predicted'); ax.set_ylabel('empirical')
ax.legend(frameon=False, loc='lower right', fontsize=9)
plt.show()

In [ ]:
# brier
for name, p in probs.items():
    print(f'  {name:<12} {np.mean((p - yeval)**2):.4f}')

TODO try different nu and see if platt stop looking like a step. probly not but worth a check.